# 🌌 SASA Mobility Masterclass: The Definitive ML Blueprint
### Holistic Data Science for Smart City Governance & Resilience

---

## 📖 Introduction
Welcome to the most detailed analytics framework built for the **SASA Data Warehouse**. This document is a complete, executable technical guide for building, validating, and explaining high-performance Machine Learning models. 

**Project Context**: Analyzing urban mobility (Paris, Lyon, Marseille...) to optimize safety, pollution, and traffic.

## 🛠️ A — Data Preparation & Feature Engineering

### 1. The SASA Data Engine
We generate and clean a multimodal dataset representing Transport stops (GPS), Air pollutants (PM2.5), and Traffic.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

# Simulation: 1500 observations
n = 1500
df = pd.DataFrame({
    'city': np.random.choice(['Paris', 'Lyon', 'Marseille', 'Lille', 'Nice'], n),
    'station_type': np.random.choice(['Metro', 'Bus', 'Tram'], n),
    'lat': 45 + np.random.normal(0, 2, n),
    'lon': 2 + np.random.normal(0, 2, n),
    'connections': np.random.randint(1, 12, n),
    'daily_traffic': np.random.exponential(40000, n) + 5000,
    'pm25': np.random.normal(18, 6, n),
    'no2': np.random.normal(35, 12, n)
})

# Introduce missing values for cleaning demonstration
df.loc[np.random.choice(df.index, 50), 'pm25'] = np.nan
df.loc[np.random.choice(df.index, 30), 'daily_traffic'] = np.nan

# Clean & engineer
df['pm25'].fillna(df['pm25'].median(), inplace=True)
df['daily_traffic'].fillna(df['daily_traffic'].median(), inplace=True)

df['pollution_idx'] = df['pm25'] + df['no2']
display(df.head())

## 🧠 B — Scientific Foundation & Model Intuition

| Algorithm | Intuition & Logic | Assumptions | Limitations |
|-----------|-------------------|-------------|-------------|
| **Random Forest** | Ensemble of Decision Trees via Bagging. Reduces variance. | Non-linear relationships. | Extrapolation outside training data. |
| **Logistic Regression**| Linear decision boundary using sigmoid function. | Linearity of independent variables & log-odds. | Cannot capture complex non-linear limits. |
| **XGBoost** | Sequential gradient boosting of weak trees. Corrects past errors. | High quality numeric/tabular data. | Prone to overfitting without tuning. |
| **K-Means** | Centroid optimization minimizing Variance (WCSS). | Spherical, equally sized clusters. | Struggles with varying density/noise. |
| **SARIMA** | Forecasting using Auto-Regressive, Moving Average & Seasonality | Stationarity (constant mean & variance) | Requires long historical history. |

## 📊 C — Classification: Environmental Alerts
**Goal**: Classify if a station is at **'Risk'** (Hazard Alert: PM2.5 > 20).

**Implementation**: Random Forest Pipeline vs Logistic Regression Pipeline + GridSearch.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.model_selection import GridSearchCV

df['hazard'] = (df['pm25'] > 20).astype(int)
X_clf = df.drop(['hazard', 'pm25', 'pollution_idx'], axis=1)
y_clf = df['hazard']

num_features = ['lat', 'lon', 'connections', 'no2', 'daily_traffic']
cat_features = ['city', 'station_type']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

# Model 1: Logistic Regression Pipeline
log_pipe = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(class_weight='balanced'))])
log_pipe.fit(X_train, y_train)
y_pred_log = log_pipe.predict(X_test)
y_prob_log = log_pipe.predict_proba(X_test)[:, 1]

# Model 2: Random Forest Pipeline
rf_pipe = Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(random_state=42, class_weight='balanced'))])
rf_grid = GridSearchCV(rf_pipe, param_grid={'clf__n_estimators': [50, 100], 'clf__max_depth': [5, 10]}, cv=3, scoring='f1')
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

# Results Comparison
def print_metrics(y_true, y_pred, y_prob, name):
    print(f"\n--- {name} ---")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred):.3f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.3f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred):.3f}")
    print(f"ROC AUC:   {roc_auc_score(y_true, y_prob):.3f}")

print_metrics(y_test, y_pred_log, y_prob_log, "Logistic Regression")
print_metrics(y_test, y_pred_rf, y_prob_rf, "Random Forest (Tuned)")

# Visualization: ROC Curve & Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fpr_log, tpr_log, _ = roc_curve(y_test, y_prob_log)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
axes[0].plot(fpr_log, tpr_log, label='LogReg')
axes[0].plot(fpr_rf, tpr_rf, label='Random Forest')
axes[0].plot([0,1],[0,1], 'k--')
axes[0].set_title('ROC Curve')
axes[0].legend()

sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', ax=axes[1], cmap='Blues')
axes[1].set_title('Confusion Matrix (Random Forest)')
plt.show()

## 📉 D — Regression: Demand & Traffic Forecasting
**Goal**: Predict daily traffic intensity based on infrastructure footprint.
**Comparison**: **Ridge Regression** vs **XGBoost Regressor**.

In [ ]:
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_reg = df['daily_traffic']
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_clf, y_reg, test_size=0.2, random_state=42)

# Model 1: Ridge
ridge_pipe = Pipeline([('prep', preprocessor), ('reg', Ridge(alpha=1.0))]).fit(X_train_r, y_train_r)
pred_ridge = ridge_pipe.predict(X_test_r)

# Model 2: XGBoost Regressor
xgb_pipe = Pipeline([('prep', preprocessor), ('reg', XGBRegressor(n_estimators=100, learning_rate=0.1))]).fit(X_train_r, y_train_r)
pred_xgb = xgb_pipe.predict(X_test_r)

# Metrics
for name, pred in [('Ridge', pred_ridge), ('XGBoost', pred_xgb)]:
    print(f"\n--- {name} ---")
    print(f"MSE:  {mean_squared_error(y_test_r, pred):.2f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, pred)):.2f}")
    print(f"MAE:  {mean_absolute_error(y_test_r, pred):.2f}")
    print(f"R2:   {r2_score(y_test_r, pred):.3f}")

# Visualization: Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test_r, pred_xgb, alpha=0.5, color='orange', label='Predictions')
plt.plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], 'r--', label='Perfect Fit')
plt.title('XGBoost: Actual vs Predicted Traffic')
plt.xlabel('Actual Traffic')
plt.ylabel('Predicted Traffic')
plt.legend()
plt.show()

## 🧬 E — Clustering: Spatial Hub Segmentation
**Goal**: Identify station clusters using **K-Means** vs **Hierarchical Clustering**.
**Evaluation**: Silhouette Score, Davies-Bouldin, Elbow Method.

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

X_clust = StandardScaler().fit_transform(df[['lat', 'lon', 'daily_traffic', 'pollution_idx']])

# Elbow Method for K-Means
inertias = []
for k in range(2, 8):
    inertias.append(KMeans(n_clusters=k, random_state=42).fit(X_clust).inertia_)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(range(2, 8), inertias, marker='o')
plt.title('Elbow Method (K-Means)')

# Fit chosen K (e.g. 4)
km = KMeans(n_clusters=4, random_state=42).fit(X_clust)
agg = AgglomerativeClustering(n_clusters=4).fit(X_clust)

print(f"K-Means Silhouette: {silhouette_score(X_clust, km.labels_):.3f} | Davies-Bouldin: {davies_bouldin_score(X_clust, km.labels_):.3f}")
print(f"Agglom Silhouette: {silhouette_score(X_clust, agg.labels_):.3f} | Davies-Bouldin: {davies_bouldin_score(X_clust, agg.labels_):.3f}")

# PCA Visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_clust)
plt.subplot(1, 2, 2)
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=km.labels_, palette='viridis')
plt.title('K-Means Clusters (PCA 2D)')
plt.show()

## 📈 F — Time Series Forecasting
**Goal**: Forecast urban traffic. Compare **SARIMA** (Statistical) with **XGBoost TS** (Machine Learning).
**Analysis**: ADF Stationarity, Seasonal Decomposition.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Generate mock daily TS data
dates = pd.date_range('2024-01-01', periods=365*2)
seasonality = 1000 * np.sin(2 * np.pi * np.arange(len(dates)) / 7)
trend = 10 * np.arange(len(dates))
noise = np.random.normal(0, 500, len(dates))
ts_data = pd.Series(50000 + trend + seasonality + noise, index=dates)

# 1. ADF Test
adf_res = adfuller(ts_data)
print(f"ADF Statistic: {adf_res[0]:.3f}, p-value: {adf_res[1]:.4f}")

seasonal_decompose(ts_data, period=7).plot()
plt.show()

# 2. Splitting
train_ts, test_ts = ts_data[:-30], ts_data[-30:]

# 3. SARIMA
sarima = SARIMAX(train_ts, order=(1,1,1), seasonal_order=(0,0,0,0)).fit(disp=False)
pred_sarima = sarima.forecast(steps=30)

# 4. XGBoost for TS (via Lags)
df_ts = pd.DataFrame({'y': ts_data})
df_ts['lag_1'] = df_ts['y'].shift(1)
df_ts['lag_7'] = df_ts['y'].shift(7)
df_ts.dropna(inplace=True)
train_xgb = df_ts.iloc[:-30]
test_xgb = df_ts.iloc[-30:]

xgb_ts = XGBRegressor(n_estimators=50).fit(train_xgb[['lag_1', 'lag_7']], train_xgb['y'])
pred_xgb_ts = xgb_ts.predict(test_xgb[['lag_1', 'lag_7']])

# Evaluation
def mape(y_t, y_p): return np.mean(np.abs((y_t - y_p) / y_t)) * 100

print(f"SARIMA - RMSE: {np.sqrt(mean_squared_error(test_ts, pred_sarima)):.2f}, MAPE: {mape(test_ts.values, pred_sarima.values):.2f}%")
print(f"XGB TS - RMSE: {np.sqrt(mean_squared_error(test_xgb['y'], pred_xgb_ts)):.2f}, MAPE: {mape(test_xgb['y'].values, pred_xgb_ts):.2f}%")

plt.figure(figsize=(12, 4))
plt.plot(test_ts.index, test_ts.values, label='Actual')
plt.plot(test_ts.index, pred_sarima.values, label='SARIMA', linestyle='--')
plt.plot(test_ts.index, pred_xgb_ts, label='XGBoost', linestyle='-.')
plt.legend()
plt.title('Time Series Forecast Comparison (30 Days)')
plt.show()

## 🚀 G — Advanced AI: Deep Learning & NLP
### Deep Learning Regression (PyTorch/Keras conceptual architecture)

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp_pipe = Pipeline([('prep', preprocessor), ('mlp', MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=200))])
mlp_pipe.fit(X_train_r, y_train_r)
pred_mlp = mlp_pipe.predict(X_test_r)
print(f"Advanced Deep Neural Network (MLP) trained. R2 Score: {r2_score(y_test_r, pred_mlp):.3f}")

### Sentiment Analysis (NLP) for Feedback
Analyzes public sentiment on mobility updates.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

reviews = ["Train was extremely late and dirty", "Fast and reliable transport", "Horrible traffic near the station", "Great bus network!"]
labels = [0, 1, 0, 1]  # 0=Negative, 1=Positive

tfidf = TfidfVectorizer()
X_nlp = tfidf.fit_transform(reviews)
nb = MultinomialNB().fit(X_nlp, labels)

new_review = ["The new metro line is amazing and fast"]
print(f"Sentiment Prediction for '{new_review[0]}': {'Positive' if nb.predict(tfidf.transform(new_review))[0] == 1 else 'Negative'}")

---
# 🏁 Conclusion
This master template fulfills all requirements for SASA Mobility, successfully bridging robust feature engineering, advanced evaluation metrics (AUC, MAE, Silhouette, MAPE), and state-of-the-art multi-model predictive architectures.